# Broken Stitch Detection — Original-data audit and staged GPU evaluation

**ICEEICT 2027 | Reconstruction protocol v2**

This notebook downloads the 2.15 GB `dataset.zip` **from your own Google Drive account inside Colab**. It first creates a ZIP inventory and image audit. **Training is disabled by default** until a human has verified the label and `source_id` lineage of the photographs and all Photoshop/augmentation descendants. Never use smoke-test scores as scientific results.

Original thesis reported 181 initial images, but the number of independent source photographs, real defects and augmented variants remains unverified. In particular, the thesis reports 20,000 / 10,000 / 9,134 image counts for different stages.

**Run in Colab → Runtime → Change runtime type → GPU.** Grant access to *your* Drive when prompted. Intermediate files stay in Colab; small audit reports are copied into your existing Drive research folder.

In [ ]:
# Cell 1 — environment and Drive authentication. No dataset access without your approval.
import os, sys, json, re, csv, shutil, pathlib, zipfile, collections, hashlib, time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import torch
print('Python',sys.version.split()[0], 'PyTorch',torch.__version__)
print('GPU available?',torch.cuda.is_available())
if torch.cuda.is_available():print('GPU',torch.cuda.get_device_name(0))
auth.authenticate_user()
drive_api=build('drive','v3',cache_discovery=False)

FOLDER_ID='1KjVGEu_g4VlrNh1IODWs4hIDnXqR8SVH'
DATASET_ID='1vE9D1jv_fmj1kNPejgTHgGCAfA_6EjPO'
CODE_ID='1Fq6USOufi3YGAKc6TdigF_X0bfHB3Wuf'
BASE=pathlib.Path('/content/brokenstitch_eval'); BASE.mkdir(exist_ok=True,parents=True)
print('Workspace:',BASE)

In [ ]:
# Cell 2 — download using the authenticated Drive API, not the limited ChatGPT connector.
def download_drive(file_id, dest):
    dest=pathlib.Path(dest);dest.parent.mkdir(exist_ok=True,parents=True)
    meta=drive_api.files().get(fileId=file_id, fields='id,name,size,mimeType',supportsAllDrives=True).execute()
    size=int(meta.get('size','0'))
    if dest.exists() and dest.stat().st_size==size:
        print('Verified existing local download:',dest, size);return dest
    if dest.exists():dest.unlink()  # Avoid mistakenly appending to a corrupt/partial file.
    free=shutil.disk_usage(dest.parent).free
    if size and free < size+2_000_000_000:
        raise RuntimeError(f'Not enough free disk for download plus processing: {free/1e9:.1f} GB')
    print(f'Downloading {meta["name"]}: {size/1e9:.2f} GB to {dest}')
    request=drive_api.files().get_media(fileId=file_id,supportsAllDrives=True)
    with open(dest,'wb') as f:
        downloader=MediaIoBaseDownload(f,request,chunksize=32*1024*1024)
        done=False
        while not done:
            status,done=downloader.next_chunk(num_retries=5)
            if status: print(f'  {status.progress()*100:.1f}% ',end='')
    print('\nCompleted:',dest.stat().st_size,'bytes')
    if size and dest.stat().st_size!=size:raise IOError('Downloaded file size differs from Google Drive metadata')
    return dest

archive_path=download_drive(DATASET_ID, BASE/'dataset.zip')

In [ ]:
# Cell 3 — archive-only audit (NO training; does not decompress images).
IMAGE_EXTS={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
LABEL_ALIASES={'actual':'actual','correct':'actual','normal':'actual','good':'actual',
               'faulty':'faulty','wrong':'faulty','broken':'faulty','defect':'faulty','defective':'faulty'}
def inferred_label(path):
    parts=[s.lower() for s in pathlib.PurePosixPath(path).parts[:-1]]
    kinds={LABEL_ALIASES[p] for p in parts if p in LABEL_ALIASES}
    return next(iter(kinds)) if len(kinds)==1 else 'unresolved'

def audit_zip(path, outdir):
    outdir=pathlib.Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    if not zipfile.is_zipfile(path):raise ValueError('Not a valid ZIP archive')
    rows=[];names=set();unsafe=[];duplicates=[]
    with zipfile.ZipFile(path) as z:
        for x in z.infolist():
            name=x.filename.replace('\\','/')
            p=pathlib.PurePosixPath(name)
            if (p.is_absolute() or '..' in p.parts or re.match(r'^[a-zA-Z]:',name)):
                unsafe.append(name)
            if name in names:duplicates.append(name)
            names.add(name)
            extension=pathlib.PurePosixPath(name).suffix.lower()
            rows.append({'member':name,'compressed_bytes':x.compress_size,
                         'uncompressed_bytes':x.file_size,'extension':extension,
                         'label_from_folders':inferred_label(name),'is_dir':x.is_dir(),
                         'encrypted':bool(x.flag_bits & 1)})
    import pandas as pd
    df=pd.DataFrame(rows)
    images=df[(~df.is_dir)&df.extension.isin(IMAGE_EXTS)]
    summary={'zip_bytes':pathlib.Path(path).stat().st_size,'zip_entries':len(df),
      'image_entries':len(images),'uncompressed_total_bytes':int(df.uncompressed_bytes.sum()),
      'image_label_folder_counts':images.label_from_folders.value_counts().to_dict(),
      'top_directories':df.member.str.split('/').str[0].value_counts().head(20).to_dict(),
      'unsafe_entries':unsafe[:50],'duplicate_member_names':duplicates[:50],
      'encrypted_entries':int(df.encrypted.sum()),'status':'METADATA_ONLY; lineage not verified'}
    df.to_csv(outdir/'zip_entries.csv',index=False)
    (outdir/'zip_report.json').write_text(json.dumps(summary,indent=2),encoding='utf8')
    print(json.dumps(summary,indent=2))
    return df,summary

zip_entries,zip_summary=audit_zip(archive_path,BASE/'audit')
if zip_summary['unsafe_entries'] or zip_summary['duplicate_member_names'] or zip_summary['encrypted_entries']:
    raise RuntimeError('Unsafe, ambiguous or encrypted ZIP members. STOP and inspect report.')
if not zip_summary['image_entries']:raise RuntimeError('No images in archive; inspect zip_entries.csv')

In [ ]:
# Cell 4 — sample photographs directly from ZIP; small human-review preview, not proof of source lineage.
from PIL import Image,ImageOps,ImageDraw
import io,random
with zipfile.ZipFile(archive_path) as z:
    groups={k:[] for k in ('actual','faulty','unresolved')}
    for x in z.infolist():
        if not x.is_dir() and pathlib.PurePosixPath(x.filename).suffix.lower() in IMAGE_EXTS and x.file_size<25_000_000:
            groups[inferred_label(x.filename)].append(x)
    rng=random.Random(42)
    choices=[]
    for k,items in groups.items():
        choices += [(k,x) for x in rng.sample(items,min(8,len(items)))]
    if not choices:raise RuntimeError('No previewable images below 25 MB')
    w,h=240,210;cols=4;rows=(len(choices)+cols-1)//cols
    sheet=Image.new('RGB',(cols*w,rows*h),'white');draw=ImageDraw.Draw(sheet)
    for i,(k,x) in enumerate(choices):
        try:
            im=Image.open(io.BytesIO(z.read(x)))
            im=ImageOps.exif_transpose(im).convert('RGB')
            im.thumbnail((w-14,h-54))
            xx=(i%cols)*w; yy=(i//cols)*h
            sheet.paste(im,(xx+(w-im.width)//2,yy+10))
            draw.text((xx+6,yy+h-38),k,fill='black')
            draw.text((xx+6,yy+h-20),pathlib.PurePosixPath(x.filename).name[:35],fill='black')
        except Exception as e:print('Preview decode failed:',x.filename,type(e).__name__)
    preview=BASE/'audit'/'contact_sheet.jpg';sheet.save(preview,quality=88)
    display(sheet)

In [ ]:
# Cell 5 — copy small reports to the original Drive folder so ChatGPT can inspect them.
def upload_report(path,folder_id=FOLDER_ID):
    path=pathlib.Path(path)
    metadata={'name':'BS_AUDIT_2026_'+path.name,'parents':[folder_id]}
    media=MediaFileUpload(str(path),mimetype='text/csv' if path.suffix=='.csv' else ('image/jpeg' if path.suffix=='.jpg' else 'application/json'),resumable=True)
    result=drive_api.files().create(body=metadata,media_body=media,fields='id,name,webViewLink',supportsAllDrives=True).execute()
    print('Uploaded:',result.get('name'),result.get('webViewLink',result['id']))
    return result

uploaded=[]
for p in [BASE/'audit'/'zip_report.json',BASE/'audit'/'zip_entries.csv',BASE/'audit'/'contact_sheet.jpg']:
    uploaded.append(upload_report(p))
print('Stage-1 preliminary audit uploaded to Drive. Source/group audit still required.')

## Stage 2: extract images, create verified manifest

**Stop if the ZIP inventory reveals unexpected directory structure, malformed image files, or unacceptable storage consumption.** Extraction writes only beneath `/content/brokenstitch_eval/data`. Follow this with manual parent-image/source-label verification. No experiment result should be reported yet.

In [ ]:
# Cell 6 — guarded extraction and official rebuilt source download.
expanded=zip_summary['uncompressed_total_bytes']; free=shutil.disk_usage(BASE).free
if expanded>30*1024**3:raise RuntimeError(f'Archive expands to {expanded/1e9:.1f} GB; review before extraction')
if expanded>free*0.85:raise RuntimeError(f'Insufficient free disk for {expanded/1e9:.1f} GB extraction')
rawdir=BASE/'data'/'raw';rawdir.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive_path) as z:
    root=rawdir.resolve()
    for x in z.infolist():
        member=x.filename.replace('\\','/')
        dest=(rawdir/member).resolve()
        if not (dest==root or root in dest.parents):raise RuntimeError(f'Unsafe member path: {member}')
    z.extractall(rawdir)
print('Images extracted beneath',rawdir)
code_zip=download_drive(CODE_ID,BASE/'source.zip')
code_dir=BASE/'source';code_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(code_zip) as z:z.extractall(code_dir)
project=code_dir/'BrokenStitch_Rebuild_ICEEICT2027'
sys.path.insert(0,str(project))
from broken_stitch.data import inspect_to_csv
manifest_path=BASE/'audit'/'inventory.csv'
inv=inspect_to_csv(str(rawdir),str(manifest_path))
print('Total decoded/inventoried:',len(inv),'decode failures:',int((inv.decode_error!='').sum()))
print('Label counts:',inv.label.value_counts().to_dict())
print('Missing source IDs:',int((inv.source_id=='').sum()))
upload_report(manifest_path)

### Mandatory human gate

Open `BS_AUDIT_2026_inventory.csv` in Drive. Every row needs a *verified* `source_id` identifying the real photograph or parent. The Photoshop defect and each augmentation of that photograph must share its `source_id`, **even when normal and faulty labels differ**. Labels must be `actual` or `faulty`. Set `is_synthetic` honestly (`yes`, `no` or `unknown`).

**Do not fake source IDs or simply use one ID per filename.** A split without known parentage is not a reliable scientific test. Upload the verified inventory as `BS_verified_lineage.csv` into the Drive research folder, then run Stage 3. For some datasets, this audit may require collecting more independently photographed examples.

If you only need the quick ZIP report, stop after Cell 5 and share the Drive report links with ChatGPT.

In [ ]:
# Cell 7 — look for a manually audited CSV in Drive; fail closed otherwise.
LINEAGE_FILENAME='BS_verified_lineage.csv'
q="name = '"+LINEAGE_FILENAME.replace("'","\'")+"' and trashed = false"
found=drive_api.files().list(q=q,fields='files(id,name,parents),nextPageToken',page_size=100,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()['files']
found=[f for f in found if FOLDER_ID in f.get('parents',[])]
if len(found)!=1:
    print(f'HALTED: Expected one {LINEAGE_FILENAME} in research folder; found {len(found)}. No training.')
    LINEAGE_OK=False
else:
    lineage_path=download_drive(found[0]['id'],BASE/LINEAGE_FILENAME)
    from broken_stitch.data import load_manifest
    lineage_df=load_manifest(str(lineage_path))
    if 'is_synthetic' not in lineage_df or lineage_df.is_synthetic.eq('unknown').any():
        print('Synthetic flags unknown; record limitations in results.')
    print('Rows',len(lineage_df),'verified source-group IDs supplied',lineage_df.source_id.nunique())
    print('Need manual confirmation that every source_id is correct.')
    LINEAGE_OK=True

## Stage 3: GPU training — disabled until human source verification

The code path below runs **only** if the inventory has been verified by a qualified researcher and `RUN_TRAINING` is deliberately set to `True`. **Validation** chooses the preferred model; the **test** set should be evaluated only after the model and threshold are frozen. Unverified ZIP metadata alone cannot authorize training.

In [ ]:
# Cell 8 — LOCKED: Requires human-verification statement and GPU.
RUN_TRAINING=False  # Set True only after source IDs, labels, and image rights have been checked.
RESEARCHER_CONFIRMS_LINEAGE=False  # Explicit, accountable human verification.
if not (RUN_TRAINING and RESEARCHER_CONFIRMS_LINEAGE and globals().get('LINEAGE_OK',False)):
    print('Training correctly halted: human-reviewed lineage is not yet verified.')
else:
    if not torch.cuda.is_available():raise RuntimeError('GPU runtime required for full training')
    from broken_stitch.data import create_group_split,audit_manifest
    from broken_stitch.engine import train,evaluate
    split_path=BASE/'audit'/'source_disjoint_split.csv'
    create_group_split(str(lineage_path),str(split_path),seed=42)
    audit=audit_manifest(str(split_path),check_images=True,hash_all=True)
    print('SOURCE DISJOINT AUDIT:',audit)
    runs=BASE/'runs';runs.mkdir(exist_ok=True)
    # A preliminary comparison; do not tune on test.
    cnn=train(str(split_path),str(runs/'cnn_relu'),model_name='cnn',seed=42,epochs=30,batch_size=32,device='cuda')
    print('CNN validation best:',cnn['best_macro_f1'])
    vgg=train(str(split_path),str(runs/'vgg_finetune'),model_name='vgg_finetune',seed=42,
              epochs=30,batch_size=16,lr=1e-5,allow_download=True,device='cuda')
    print('VGG validation best:',vgg['best_macro_f1'])
    print('Evaluate once on untouched test AFTER model choice is frozen; no test-based model selection.')

In [ ]:
# Cell 9 — independent final test. Run ONCE only after a model is frozen using validation.
RUN_FINAL_TEST=False  # Enable only when Stage 3 completed and model selection is finalized.
FROZEN_MODEL='vgg_finetune'  # Explicitly select from ['cnn_relu','vgg_finetune'] using validation only.
if not (RUN_FINAL_TEST and RUN_TRAINING and RESEARCHER_CONFIRMS_LINEAGE and globals().get('LINEAGE_OK',False)):
    print('Independent test held out. No test accuracy claimed.')
else:
    from broken_stitch.engine import evaluate
    selected=(BASE/'runs'/FROZEN_MODEL/'best.pt')
    if not selected.is_file():raise FileNotFoundError(selected)
    out=BASE/'audit'/'locked_test_results.json'
    report=evaluate(str(selected),str(split_path),part='test',output=str(out))
    print(json.dumps(report,indent=2))
    upload_report(out)
    upload_report(split_path)
    print('Test results uploaded. Do NOT iterate on model choices using this test.')

## What to send back

After Stage 1, the research folder will contain small files named:
- `BS_AUDIT_2026_zip_report.json`
- `BS_AUDIT_2026_zip_entries.csv`
- `BS_AUDIT_2026_contact_sheet.jpg`

After Stage 2, it will also contain `BS_AUDIT_2026_inventory.csv`. Send a message saying **“audit uploaded”**. We can then inspect the actual dataset structure and update the paper plan. Training accuracy should not be expected from an archive listing alone.

**Note:** This notebook was authored without accessing the contents of your original ZIP, and has not been run in Colab. Its functionality is partly testable with toy ZIPs, but Drive/Colab authorization and the 2.15 GB dataset path can only be tested when the user runs it.